# Implementing a Simple LLM Call Chain

## Introduction

In this tutorial, you'll learn how to implement a **simple LLM call chain** using the OpenAI SDK directly (without frameworks like LangChain).

### What is an LLM Chain?

An LLM chain is a sequence of operations where:
- One step's output feeds into the next step
- Multiple LLM calls are orchestrated together
- Complex, multi-step workflows are enabled

### What You'll Build

A **writing improvement chain** that follows this pattern:
1. **Generate** - Create a first draft about a topic
2. **Critique** - Analyze what could be improved
3. **Improve** - Rewrite incorporating the feedback

### Prerequisites

- Python 3.12+
- OpenAI API key (set in a `.env` file)
- Basic understanding of functions and string formatting

### Learning Objectives

By the end of this notebook, you will:
- Understand how to chain multiple LLM calls together
- Create a reusable wrapper function for OpenAI API calls
- Implement a three-step writing improvement workflow
- See how intermediate outputs flow through a chain

## Setup: Import Libraries and Load API Key

First, we'll import the necessary libraries and load the OpenAI API key from a `.env` file.

**Note**: Make sure you have a `.env` file in your project directory with the following content:
```
OPENAI_API_KEY=your_api_key_here
```

In [1]:
# Install required packages (uncomment if needed)
# !pip install openai python-dotenv

from openai import OpenAI
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("Setup complete! OpenAI client initialized.")

Setup complete! OpenAI client initialized.


## Create a Reusable LLM Wrapper Function

Before building our chain, we'll create a simple helper function that wraps OpenAI API calls. This makes our code cleaner and more maintainable.

The `llm_call()` function:
- Takes a prompt as input
- Sends it to GPT-4
- Returns the text response

This abstraction allows us to focus on chain logic rather than API details.

In [2]:
def llm_call(prompt):
    """
    Simple wrapper for OpenAI API calls.
    
    Args:
        prompt (str): The prompt to send to the LLM
        
    Returns:
        str: The LLM's response text
    """
    response = client.chat.completions.create(
        model="gpt-5",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# Test the function with a simple prompt
test_response = llm_call("Say 'Hello, World!' in a creative way.")
print("Test response:", test_response)

Test response: - Pixels wake and glow
  Across a wired horizon—
  Hello, World!
- 👋🌍
- Morse: .... . .-.. .-.. --- --..-- / .-- --- .-. .-.. -.. -.-.--
- Base64: SGVsbG8sIFdvcmxkIQ==
- Pig Latin: Ellohay, Orldway!
- A global chorus: Hello, World! Bonjour, Monde! Hola, Mundo! こんにちは、世界！
- From stardust through fiber: Hello, World!


## Implement the Writing Improvement Chain

Now we'll implement the core chain that demonstrates how multiple LLM calls work together.

### The Three-Step Process

1. **Generate Draft**: Create initial content about the topic
2. **Critique Draft**: Analyze what could be improved (clarity, structure, tone, etc.)
3. **Improve Draft**: Rewrite incorporating the critique

Notice how each step uses the output from the previous step - this is the essence of chaining.

In [8]:
def writing_improvement_chain(topic):
    """
    A three-step chain that generates, critiques, and improves writing.
    
    Args:
        topic (str): The topic to write about
        
    Returns:
        dict: Contains 'draft', 'critique', and 'final' versions
    """
    print(f"Starting writing improvement chain for topic: '{topic}'\n")
    print("=" * 70)
    
    # Step 1: Generate first draft
    print("\nSTEP 1: Generating initial draft...")
    draft = llm_call(f"Write a paragraph about {topic}")
    print(f"\nDraft:\n{draft}")
    
    # Step 2: Critique the draft
    print("\n" + "=" * 70)
    print("\nSTEP 2: Analyzing draft for improvements...")
    critique = llm_call(f"What could be improved in this paragraph? Be specific. Do not rewrite the content, only output 3 improvements.\n\n{draft}")
    print(f"\nCritique:\n{critique}")
    
    # Step 3: Rewrite with improvements
    print("\n" + "=" * 70)
    print("\nSTEP 3: Rewriting with improvements...")
    final = llm_call(f"Rewrite this paragraph, keep it the same size as original and incorporate the following feedback:\n\nOriginal:\n{draft}\n\nFeedback:\n{critique}")
    print(f"\nFinal Version:\n{final}")
    
    print("\n" + "=" * 70)
    print("\nChain complete!\n")
    
    return {
        "draft": draft,
        "critique": critique,
        "final": final
    }

print("writing_improvement_chain() function defined successfully!")

writing_improvement_chain() function defined successfully!


## Example: Write About Artificial Intelligence

Let's test our chain with a technical topic. Watch how the chain:
1. Creates an initial draft
2. Identifies areas for improvement
3. Produces a refined final version

In [9]:
result = writing_improvement_chain("artificial intelligence")

Starting writing improvement chain for topic: 'artificial intelligence'


STEP 1: Generating initial draft...

Draft:
Artificial intelligence (AI) refers to computer systems designed to perform tasks that typically require human intelligence, such as recognizing patterns, understanding language, making decisions, and learning from data. Powered largely by machine learning and deep learning, AI systems excel at finding subtle relationships in vast datasets and can outperform humans in narrow domains like image recognition or game playing. AI is already embedded in everyday life—from recommendation engines and virtual assistants to medical diagnostics, fraud detection, and autonomous vehicles—driving efficiency and new capabilities across industries. At the same time, AI raises important challenges, including bias and fairness, transparency, privacy, security, and potential impacts on jobs and power dynamics. Ongoing research, standards, and regulation aim to ensure AI is reliable, safe,